# 7. Explaining a run

The earlier notebooks drive the pieces: the scan kernel, the physics, the score shapes.
This one drives **the whole pipeline** — screen, scan, score, clean, label, pack, write
— as an ordinary Python call, and then reads the result properly.

Two searches are run below — one that finds ground and one that finds none — because
the summary of an empty result is the one that matters most and is the easiest to
neglect. The full Arequipa DEM has its own notebook,
**[8](08_the_full_dem.ipynb)**.

Three things are worth knowing before the first call:

- **Everything the command line can do, the library can do.** Configuration files,
  the memory pre-flight, the run summary. There is no CLI-only behaviour left.
- **The pipeline returns its results.** It used to return `None` and leave callers to
  find and re-read the JSON it had just written.
- **It explains itself.** A plain-language account of what was found and why, printed
  and saved as `explanation.txt`, on by default.

In [1]:
import numpy as np

import contextlib
import io
import os
import tempfile

import oroscope
from oroscope import explain, site_searcher as ss

WORK = tempfile.mkdtemp(prefix="oroscope_nb07_")
print("working in", WORK)

working in /tmp/oroscope_nb07_qo_dqism


## Configuration is data, not a command-line concern

`default_config()` returns every knob the tool understands, with its default.
`generate_config(path, preset)` writes that as a template, and `load_config(path)` reads
one back. All three used to exist only inside `main()`, reachable by running the CLI.

A template naming **every** key matters more than it sounds: a config with holes in it
falls back silently for whatever it omits, and the fallback file is the least visible
input the tool has.

In [2]:
cfg = ss.default_config("arequipa")
print(f"{len(cfg)} keys, e.g.:")
for key in ("dem_path", "min_slope_deg", "max_slope_deg", "candidate_stride",
            "downsample_factor", "min_score", "explain"):
    print(f"   {key:>20}: {cfg[key]!r}")

path = os.path.join(WORK, "arequipa.json")
ss.generate_config(path, "arequipa")
print(f"\nwritten and read back identically: {ss.load_config(path) == cfg}")

79 keys, e.g.:
               dem_path: 'arequipa_SRTMGL1.tif'
          min_slope_deg: 3.0
          max_slope_deg: 25.0
       candidate_stride: 5
      downsample_factor: 4
              min_score: 0.0
                explain: True

written and read back identically: True


## Before a big run: what will it cost?

`estimate_peak_memory_gb` predicts the *anonymous* allocations from the DEM's size and
two parameters. The memory-mapped DEM is deliberately excluded — it is file-backed and
the kernel can evict it, and counting it would make every large search look impossible
when the streaming design exists precisely so that it is not.

`downsample_factor` is the knob that matters: the labelling arrays scale as its inverse
square. Here is the real Arequipa DEM, 10204 × 12603 pixels.

In [3]:
rows, cols = 10204, 12603
print(f"Arequipa DEM: {rows} x {cols} = {rows*cols/1e6:.0f} Mpx\n")
for ds in (1, 2, 4, 8):
    need = ss.estimate_peak_memory_gb(rows, cols, downsample_factor=ds)
    print(f"   downsample_factor {ds}:  {need:5.2f} GiB")

have = ss.available_memory_gb()
print(f"\navailable right now: {have:.1f} GiB" if have else "\n(memory not reportable here)")
print("\nThis is why the full run uses downsample_factor 4.")

Arequipa DEM: 10204 x 12603 = 129 Mpx

   downsample_factor 1:   4.45 GiB
   downsample_factor 2:   2.74 GiB
   downsample_factor 4:   2.32 GiB
   downsample_factor 8:   2.21 GiB

available right now: 7.0 GiB

This is why the full run uses downsample_factor 4.


The estimate is rough and says so — `survival_fraction` is the share of pixels passing
the topographic screen, which is terrain-dependent and unknown until the screen has run.
It is meant to catch the order-of-magnitude mistake, not to predict a number.

`preflight_memory` does the whole job: estimate, warn if it is close, and cap the
process's address space so a search that outgrows the machine fails with `MemoryError`
naming itself rather than letting the kernel's OOM killer pick a victim — which may be
your editor. A ten-point sweep once did exactly that at 6.9 GB.

## A complete run

Synthetic terrain, so this executes anywhere. A ridge with a slope in front of it: the
slope sees the ridge, and the ridge's own flank sees the terrain rising beyond.

In [4]:
def ridge_and_slope(n, cell_x):
    """A valley between a ridge and a rising slope. Closed-form, no DEM needed."""
    cols = np.arange(n, dtype=np.float64)[None, :].repeat(n, 0)
    x = cols * cell_x
    ridge = 1400.0 * np.exp(-((x - 0.30 * n * cell_x) / (0.05 * n * cell_x)) ** 2)
    rise = np.clip((x - 0.55 * n * cell_x) / (0.45 * n * cell_x), 0, 1) ** 2 * 1500.0
    return (2200.0 + ridge + rise).astype(np.float32)

In [5]:
import tifffile as tiff

grid = ss.resolve_grid_geometry("no-such-file.tif", -15.6, cell_size_deg=1/3600)
z = ridge_and_slope(700, grid.cell_size_x)

dem = os.path.join(WORK, "ridge.tif")
tiff.imwrite(dem, z, extratags=[
    (33550, "d", 3, (1/3600, 1/3600, 0.0)),                    # ModelPixelScale
    (33922, "d", 6, (0.0, 0.0, 0.0, -72.3, -15.6, 0.0)),       # ModelTiepoint
])
print("wrote a GeoTIFF carrying its own resolution and corner:", os.path.basename(dem))

wrote a GeoTIFF carrying its own resolution and corner: ridge.tif


Now the search itself. Note what is *not* passed: no origin — the DEM carries its own
corner in the tiepoint tag, and reading it removes the most error-prone input the tool
has. A supplied origin that disagrees with the file by more than ~100 m is reported
rather than silently honoured, because a wrong origin does not fail, it
mis-georeferences every output.

The run prints a great deal. It is captured here and unpacked below.

In [6]:
log = io.StringIO()
with contextlib.redirect_stdout(log), contextlib.redirect_stderr(io.StringIO()):
    results = ss.find_grand_regions_interactive(
        dem_path=dem,
        run_output_dir=os.path.join(WORK, "run"),
        # Note search_mode and grid_type. The function's own defaults are 'single'
        # and 'square'; the config template's are 'distributed' and 'hex'. Omitting
        # them here is not the same as omitting them from a config file -- see below.
        search_mode="distributed", grid_type="hex",
        target_antennas=200, min_sub_array_size=20,
        min_width_km=1.0, antenna_spacing_km=1.0,
        min_dist_km=3.0, max_dist_km=20.0,
        downsample_factor=2, tile_size=256, candidate_stride=5, num_cores=2,
    )

print(f"{len(log.getvalue().splitlines())} lines of output captured\n")
print("returned:", ", ".join(sorted(results)))

267 lines of output captured

returned: aperture, explanation, funnel, mode, output_files, parameters, provenance, regions, results, timestamp, timings_sec


### Where a default comes from

A parameter can state its default in three places — the function's signature,
`oroscope --help`, and `default_config()` — and **all three agree**. They did not
always: ten parameters disagreed, so omitting one meant different things depending on
which door you came in by. An earlier draft of this notebook omitted `search_mode`,
quietly ran a *single* search with a 30 km minimum distance, and found nothing at all
on this small ridge. The funnel said so plainly, which is the system working, but the
trap should not have existed.

It cannot come back: a test compares the three sources pairwise, for every parameter.

Starting from `default_config()` and overriding is still the clearer habit, because it
puts every knob in front of you rather than leaving them implicit.

In [7]:
import inspect

template = oroscope.default_config()
signature = {k: v.default for k, v in
             inspect.signature(oroscope.find_grand_regions_interactive).parameters.items()
             if v.default is not inspect.Parameter.empty}

print(f"{'parameter':22} {'signature':>12} {'template':>12}")
for key in ("search_mode", "grid_type", "target_antennas", "min_dist_km",
            "min_sub_array_size", "max_road_dist_km"):
    mark = "ok" if signature[key] == template[key] else "DIFFERS"
    print(f"{key:22} {str(signature[key]):>12} {str(template[key]):>12}   {mark}")

parameter                 signature     template
search_mode             distributed  distributed   ok
grid_type                       hex          hex   ok
target_antennas               10000        10000   ok
min_dist_km                    10.0         10.0   ok
min_sub_array_size              500          500   ok
max_road_dist_km               20.0         20.0   ok


That dictionary is the same content the results JSON holds, plus the explanation and
the paths written. No re-reading the file it just wrote.

In [8]:
print(f"sites:    {results['results']['total_sites']}")
print(f"capacity: {results['results']['total_capacity']}")
print(f"stages:   {', '.join(results['timings_sec'])}")
print(f"files:    {len(results['output_files'])} written")
for f in results["output_files"]:
    print("   ", os.path.basename(f))

sites:    1
capacity: 180
stages:   load_dem, topographic_screen, ray_tracing, morphology, capacity_analysis, outputs, total
files:    5 written
    oroscope_results_ridge.tif
    oroscope_results_ridge.tfw
    oroscope_results_ridge.png
    oroscope_results_ridge.json
    provenance.json


## The funnel is the diagnostic

Every filter records how many pixels survived it. When a search returns little or
nothing, **the stage where the count collapses is the constraint responsible** — and
that is the single most useful thing anyone can be told about a disappointing run.

In [9]:
for stage, count in results["funnel"].items():
    print(f"   {stage:<34} {count:>12,}")

binding = explain.binding_constraint(results["funnel"])
print(f"\nbinding constraint: {binding['stage']!r}")
print(f"   kept {100*binding['kept_fraction']:.1f}% of the {binding['before']:,} that reached it")
print(f"   change: {binding['knob']}")

   DEM pixels                              490,000
   finite elevation                        490,000
   slope 3.0-25.0 deg                      226,100
   kept by stride 5                         45,224
   directions accepted                      43,684
   after gap closing                       198,024
   after pruning (< 1.0 km wide)           194,010
   pixels in selected sites (est.)         164,820

binding constraint: 'slope 3.0-25.0 deg'
   kept 46.1% of the 490,000 that reached it
   change: min_slope_deg / max_slope_deg


Two stages are excluded from that search by construction, and it is worth knowing why:

- **`kept by stride N`** is a deliberate subsample, not a filter. It removes four
  candidates in five and the acceptance is unchanged, so calling it the constraint
  would name the same answer on nearly every run.
- **`after gap closing`** *adds* pixels. A stage that grows the set cannot be what
  shrank it.

## Which sites are actually in the result

`sites` lists everything that cleared the area and capacity thresholds. With
`stop_at_target`, selection walks that capacity-sorted list until the target is met and
stops — so the list can be longer than the result. Only the selection is in
`total_sites`, `total_capacity` and the exported raster.

Each record says which it is.

In [10]:
log2 = io.StringIO()
with contextlib.redirect_stdout(log2), contextlib.redirect_stderr(io.StringIO()):
    truncated = ss.find_grand_regions_interactive(
        dem_path=dem, run_output_dir=os.path.join(WORK, "run2"),
        target_antennas=50, min_sub_array_size=5, stop_at_target=True,
        min_width_km=1.0, antenna_spacing_km=1.0,
        min_dist_km=3.0, max_dist_km=20.0,
        downsample_factor=2, tile_size=256, candidate_stride=5, num_cores=2,
    )

chosen, shortlisted = explain.selected_sites(truncated)
print(f"listed in the file: {len(chosen) + len(shortlisted)}")
print(f"selected:           {truncated['results']['total_sites']}\n")
for site in chosen + shortlisted:
    mark = "selected" if site["selected"] else "not selected"
    print(f"   site {site['site_id']:>3}  {site['area_km2']:>8.2f} km²  "
          f"{site['capacity_exact']:>5} detectors   {mark}")

print(f"\nsumming everything listed:  {sum(s['area_km2'] for s in chosen + shortlisted):8.2f} km²")
print(f"summing the selection:      {sum(s['area_km2'] for s in chosen):8.2f} km²  <- the raster")

listed in the file: 3
selected:           1

   site   3    150.74 km²    180 detectors   selected
   site   1     12.26 km²     24 detectors   not selected
   site   2     14.71 km²     24 detectors   not selected

summing everything listed:    177.71 km²
summing the selection:        150.74 km²  <- the raster


Totalling the wrong one over-reports, which is exactly the mistake this flag exists to
prevent. The sites that were not selected are the *next best ground*, not ground that
failed — worth keeping in the file, worth excluding from the totals.

## Attribution: what held each site back

The score is a product of **named** components, each in [0, 1], and each site's record
carries the distribution of every one. Under a product the lowest component bounds the
total from above, so naming it turns "this site scored 0.34" into something actionable.

In [11]:
site = chosen[0]
scan = site["arrival_scan"]
parts = {k[len("score_"):-len("_p50")]: v for k, v in scan.items()
         if k.startswith("score_") and k.endswith("_p50") and k != "score_p50"}

print(f"site {site['site_id']}, median score {scan['score_p50']:.3f}\n")
for name, value in sorted(parts.items(), key=lambda kv: kv[1]):
    bar = "#" * int(round(value * 40))
    print(f"   {name:>14}  {value:5.3f}  {bar}")

name, value = explain.weakest_component(scan)
print(f"\nweakest: {name} at {value:.3f}")

site 3, median score 0.341

        footprint  0.447  ##################
      geomagnetic  0.865  ###################################
      solid_angle  0.888  ####################################
            depth  1.000  ########################################
         distance  1.000  ########################################
           shower  1.000  ########################################

weakest: footprint at 0.447


On the real Colca configurations this is unambiguous: `solid_angle` is the weakest
component at **15 of 15** TAMBO sites, with everything else at 1.0 except the decay term
at 0.96. So that result is set almost entirely by `solid_angle_half_sr`, whose 0.05 sr
default is a GRAND-scale value.

That is the kind of statement the components make available and a single total does not.

## How much did closing move the area?

The reported area is not the physics-accepted area: the mask is closed morphologically
before areas are measured. The published figure is 2.29× at Colca, measured against a
stride-1 control — but each run has the number in it, as closed pixels over
stride-corrected accepted pixels.

In [12]:
ratio = explain.closing_inflation(results["funnel"],
                                 results["parameters"]["candidate_stride"])
print(f"this run: closing moved the mask by {ratio:.2f}x")
print("\nOn the real configurations:")
print("   GRAND Colca  2.19x   (against 2.29x from a stride-1 control -- an independent check)")
print("   TAMBO Colca  0.53x   (a 100 m element cannot bridge the gaps stride 5 leaves,")
print("                        so its area is a LOWER bound, not an upper one)")

this run: closing moved the mask by 0.91x

On the real configurations:
   GRAND Colca  2.19x   (against 2.29x from a stride-1 control -- an independent check)
   TAMBO Colca  0.53x   (a 100 m element cannot bridge the gaps stride 5 leaves,
                        so its area is a LOWER bound, not an upper one)


## The run, explained — the whole summary, here

Everything above is assembled for you. `explain.explain_results` takes the results
dictionary and returns a string — it opens no files, runs nothing and needs no DEM, so
a run from months ago can still be explained from its JSON.

It is on by default, printed at the end of every run and saved as `explanation.txt`
beside the results, because these runs are meant to be handed to other people and a
terminal scrollback is not. `--no_explain` suppresses it.

This is the text in full. Its sections, in order:

| section | answers |
|---|---|
| **The run** | what was searched, at what resolution, by which commit |
| **The headline** | how many sites, how much area, how many detectors |
| **Where the candidates went** | the funnel, and **which constraint bound this run** |
| **From pixels to sites** | labelled regions → area threshold → capacity threshold |
| **The sites** | each one's area, capacity, facing, score and weakest criterion |
| **Why these sites qualify** | what the ground actually offers, criterion by criterion, with coordinates |
| **What energy this geometry favours** | where the geometric aperture peaks |
| **How to read these numbers** | the closing factor *for this run*, and what area is not |
| **Which of these are assumptions** | choices rather than measurements, with measured sensitivities |
| **What to try next** | concrete commands, chosen from what this run did |

In [13]:
print(results["explanation"])

 WHAT THIS SEARCH FOUND, AND WHY

THE RUN
-------
  DEM              /tmp/oroscope_nb07_qo_dqism/ridge.tif
  Origin           -15.600000, -72.300000  (auto-detected from the GeoTIFF tiepoint)
  Resolution       0.00027778°/px  =  30.7 m N-S x 29.8 m E-W
  Layout           distributed search, hex grid, 1.0 km spacing
  Finished         2026-08-16 01:30:58
  Code             commit bd27f3f on dev (dirty tree)
  DEM checksum     sha256 d53de4fdc2b72326…
  Command          /home/mbustamante/anaconda3/envs/sssearch/lib/python3.12/site-packages/ipykernel_launcher.py -f /tmp/tmp9bj73rkb.json --HistoryManager.hist_file=:memory:

THE HEADLINE
------------
  1 site covering 150.7 km², 180 detectors against a target of 200.

  Largest by capacity: site 3, 150.74 km², 180 detectors, facing W, centred
  -15.6972, -72.1444 — paste that into a map.

  The target of 200 was not reached. The funnel below says why; the shortfall
  is 20 detectors, 10% of the target.

WHERE THE CANDIDATES WENT
----------

## Provenance

Separate from the science outputs, and the answer to "what produced this number?".

In [14]:
prov = results["provenance"]
print(f"commit:   {prov['git']['commit'][:10]} on {prov['git']['branch']}"
      f"  ({'dirty' if prov['git']['dirty'] else 'clean'} tree)")
print(f"DEM:      {os.path.basename(prov['dem']['path'])}")
print(f"          sha256 {prov['dem']['sha256'][:24]}...")
print(f"          {prov['dem']['cell_size_y_m']:.2f} m N-S x {prov['dem']['cell_size_x_m']:.2f} m E-W")
print(f"python:   {prov['platform']['python']} on {prov['platform']['system']}")
print(f"packages: {', '.join(f'{k} {v}' for k, v in list(prov['packages'].items())[:4])}, ...")

commit:   bd27f3fc1b on dev  (dirty tree)
DEM:      ridge.tif
          sha256 d53de4fdc2b72326673978ad...
          30.72 m N-S x 29.77 m E-W
python:   3.12.12 on Linux 7.0.0-28-generic
packages: numpy 2.3.4, scipy 1.16.3, numba 0.62.1, tifffile 2021.7.2, ...


---

## The other outcome: a search that finds nothing

A summary of a successful run is the easy case. The one that matters is the run that
comes back empty, because that is when a reader has no idea what to change — and it is
the case a bare results file serves worst: every section is zero and nothing says why.

Here is the same terrain asked an impossible question. The distance window is moved out
past anything this ridge can offer, so no arrival direction can be accepted.

In [15]:
empty = ss.find_grand_regions_interactive(
    dem_path=dem,
    run_output_dir=os.path.join(WORK, "empty"),
    search_mode="distributed", grid_type="hex",
    target_antennas=200, min_sub_array_size=20,
    min_width_km=1.0, antenna_spacing_km=1.0,
    min_dist_km=60.0, max_dist_km=90.0,      # further than this terrain reaches
    downsample_factor=2, tile_size=256, candidate_stride=5, num_cores=2,
    explain=False,                            # composed below instead, to keep this tidy
)

print(f"sites: {empty['results']['total_sites']}")
print(f"capacity: {empty['results']['total_capacity']}\n")
for stage, count in empty["funnel"].items():
    print(f"   {stage:<34} {count:>12,}")

      Origin: -15.600000, -72.300000  (auto-detected from the GeoTIFF tiepoint)
   ⚙️  Estimated peak memory: 0.5 GiB, available 7.0 GiB
   ⚙️  Address space capped at 5.6 GiB (max_memory_gb=0 disables)

   OROSCOPE SITE SEARCH: RUN PARAMETERS
   -> DEM File: /tmp/oroscope_nb07_qo_dqism/ridge.tif
   -> Origin: -15.6, -72.3
   -> Target: 200 antennas
   -> Spacing: 1.0 km (hex grid)
   -> Min Width: 1.0 km
   -> Slope Range: 3.0° to 25.0°
   -> Target Dist: 60 - 90 km
      Arrival window: -3° to 3° in 12 bins, 9 azimuths within ±60° of aspect
      Requires: rock, min column depth 0 g/cm²
      Baseline implies tau energies 1.22e+03 - 1.84e+03 PeV
   -> Downsample Factor: 2
   -> Resolution: 0.00027778 deg/px [auto-detected from GeoTIFF]
   -> Pixel Size: 30.72 m N-S x 29.77 m E-W (at lat -15.697)
   -> Candidate Stride: every 5 px
   -> Slope Baseline: native DEM resolution
   -> Memory: Tile Size 256x256 px
   -> RFI Zones: 0 active (Numba Optimized)

   SYSTEM & RESOURCE REPORT
   -

   Scanning Topography:   0%|          | 0/9 [00:00<?, ?tile/s]

   Scanning Topography:   0%|          | 0/9 [00:00<?, ?tile/s, candidates=1,844]

   Scanning Topography:  11%|█         | 1/9 [00:00<00:00, 239.29tile/s, candidates=5,069]

   Scanning Topography:  22%|██▏       | 2/9 [00:00<00:00, 255.44tile/s, candidates=9,626]

...

   Closing: 100%|██████████| 9/9 [00:00<00:00, 326.08tile/s]

   Pruning:   0%|          | 0/9 [00:00<?, ?tile/s]

   Pruning: 100%|██████████| 9/9 [00:00<00:00, 584.59tile/s]

      Time: 0.05s

[5/6] ℹ️  Final Analysis...
      Distributed: 0 sites found.
      Total Cap: 0 (Target: 200)
      Time: 0.00s

[6/6] 💾 Saving & Visualization...


      ✅ Map saved.
      ✅ JSON Data Summary saved.
      ✅ Provenance saved.
      Time Elapsed: 0.33s

   SELECTION FUNNEL
   stage                           |          pixels |   of DEM |  of prev
   -----------------------------------------------------------------------
   DEM pixels                      |         490,000 | 100.000% |         -
   finite elevation                |         490,000 | 100.000% | 100.000%
   slope 3.0-25.0 deg              |         226,100 |  46.143% |  46.143%
   kept by stride 5                |          45,224 |   9.229% |  20.002%
   directions accepted             |               0 |   0.000% |   0.000%
   after gap closing               |               0 |   0.000% |       n/a
   after pruning (< 1.0 km wide)   |               0 |   0.000% |       n/a
   pixels in selected sites (est.) |               0 |   0.000% |       n/a

   Regions: 0 labelled -> 0 above area threshold (5,467 px) -> 0 above capacity threshold (20 antennas)

   ✅ RESULTS SU

The funnel is the whole answer, and the summary reads it: the stage where the count
reaches zero *is* the constraint, and everything downstream of it is zero for a reason
that is not its own. A stage that empties the map wins outright over any ratio below
it, which is why the report names that one rather than the largest percentage drop.

In [16]:
binding = explain.binding_constraint(empty["funnel"])
print(f"stage:      {binding['stage']}")
print(f"reached it: {binding['before']:,} pixels")
print(f"survived:   {binding['survivors']:,}")
print(f"fatal:      {binding['fatal']}")
print(f"change:     {binding['knob']}")

stage:      directions accepted
reached it: 45,224 pixels
survived:   0
fatal:      True
change:     the arrival window (elev_min_deg/elev_max_deg), the distance window (min_dist_km/max_dist_km), min_column_depth_gcm2 and min_target_slope_deg


And the summary in full. Note what it does *not* do: it does not apologise, and it does
not pad. It states the outcome, names the stage, names the parameter, and then tells you
what to try — which for an empty result is the only useful thing a report can say.

In [17]:
print(explain.explain_results(empty))

 WHAT THIS SEARCH FOUND, AND WHY

THE RUN
-------
  DEM              /tmp/oroscope_nb07_qo_dqism/ridge.tif
  Origin           -15.600000, -72.300000  (auto-detected from the GeoTIFF tiepoint)
  Resolution       0.00027778°/px  =  30.7 m N-S x 29.8 m E-W
  Layout           distributed search, hex grid, 1.0 km spacing
  Finished         2026-08-16 01:31:01

THE HEADLINE
------------
  No site met all the constraints. That is a result, not a failure: read the
  funnel below, which names the stage where the candidates ran out.

WHERE THE CANDIDATES WENT
-------------------------
  stage                           |         pixels |   of DEM |  of prev
  ----------------------------------------------------------------------
  DEM pixels                      |        490,000 | 100.000% |         -
  finite elevation                |        490,000 | 100.000% | 100.000%
  slope 3.0-25.0 deg              |        226,100 |  46.143% |  46.143%
  kept by stride 5                |         45,224 |

Compare the two summaries. The successful one describes ground; this one describes a
constraint. Both are the same function reading the same shape of dictionary — which is
the point of keeping it a pure function of the results rather than something the
pipeline prints as it goes.

## Where to go next

- **[8. The full Arequipa DEM](08_the_full_dem.ipynb)** — the run that has never been
  done, and what to look at when it is.
- **[6. Combining and sensitivity](06_combining_and_sensitivity.ipynb)** — how firm any
  of this is.

---

*Part of the [Oroscope](https://github.com/mbustama/oroscope) tutorials. Previous: [Combining and sensitivity](06_combining_and_sensitivity.ipynb). Next: [The full Arequipa DEM](08_the_full_dem.ipynb). Full API reference: [oroscope docs](https://mbustama.github.io/oroscope/functions.html).*